In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "test_pollution_20261007_152854.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("test_pollution_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
states = pd.read_csv(RESULT_CSV)
traces = pd.read_csv(TRACE_CSV)
r0 = states.iloc[0]
print(f"{RESULT_CSV.name}: {len(states)} states, {r0['f_start'] / 1e6:g}-{r0['f_stop'] / 1e6:g} MHz, "
      f"RBW {r0['rbw'] / 1e3:g} kHz, {r0['averages']:.0f} averages")

pd.DataFrame({
    "state": states["state"],
    "SMU": [f"on, {v:g} V" if on else "off" for on, v in zip(states["smu_output"], states["smu_v_set"])],
    "reference": [f"on, {f / 1e6:.4g} MHz 0-{vpp:g} V" if on else "off"
                  for on, f, vpp in zip(states["fg_output"], states["f_ref"], states["ref_vpp"])],
    "PLL supply I [mA]": states["pll_i"] * 1e3,
    "median [dBm]": states["level_median_dbm"],
    "max [dBm]": states["level_max_dbm"],
    "f at max [MHz]": states["f_at_max"] / 1e6,
    "rise over all_off, median [dB]": states["rise_median_db"],
    "rise over all_off, max [dB]": states["rise_max_db"],
    "f at max rise [MHz]": states["f_at_rise_max"] / 1e6,
}).round(3)

In [ ]:
INK, GRID = "#52514e", "#e4e3df"
# One colour per state, in switch-on order; all_off in grey as the reference.
COLORS = {"all_off": "#8a8986", "smu_0v": "#2a78d6", "smu_vdd": "#eb6834", "smu_fg": "#1baf7a"}
NAMES = {r.state: {"all_off": "all off (SMU and generator reset)",
                   "smu_0v": "SMU output on at 0 V",
                   "smu_vdd": f"SMU at {r.smu_v_set:g} V (chip powered, PLL not configured)",
                   "smu_fg": f"+ reference {r.f_ref / 1e6:.4g} MHz on"}.get(r.state, r.state)
         for r in states.itertuples()}


def freq_axis(f_axis):
    """x-axis settings for a frequency plot; f_axis \"log\" or \"lin\"."""
    return dict(type="log" if f_axis == "log" else "linear", ticksuffix="Hz", exponentformat="SI",
                minor=dict(showgrid=f_axis == "log", gridcolor="#f1f0ed"))


def in_range(df, f_start, f_stop):
    """Rows of df with freq between f_start and f_stop (None: the edge of the trace)."""
    lo = df["freq"].min() if f_start is None else f_start
    hi = df["freq"].max() if f_stop is None else f_stop
    return df[(df["freq"] >= lo) & (df["freq"] <= hi)]


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [ ]:
F_start, F_stop = None, None  # frequency range shown [Hz]; None = edge of the trace
f_axis = "lin"  # "lin" or "log"

# Averaged spectrum per state (RMS detector, power averaging), level in the trace's RBW. The peak at the lowest
# frequencies in every state is the analyzer's own LO feedthrough (0 Hz).
fig = go.Figure()
for r in states.itertuples():
    d = in_range(traces[traces["state"] == r.state], F_start, F_stop)
    fig.add_scatter(x=d["freq"], y=d["level_dbm"], mode="lines", name=NAMES[r.state],
                    line=dict(color=COLORS.get(r.state, INK), width=1.2),
                    hovertemplate=f"{r.state}<br>%{{x:.4s}}Hz: %{{y:.1f}} dBm<extra></extra>")
fig.update_layout(title=f"Spectrum per state, instruments switched on one by one (RBW {r0['rbw'] / 1e3:g} kHz, "
                        f"{r0['averages']:.0f} averages)",
                  template="plotly_white", height=550, legend=dict(orientation="h", y=-0.2),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="frequency", gridcolor=GRID, **freq_axis(f_axis))
fig.update_yaxes(title="level [dBm in RBW]", gridcolor=GRID)
save(fig, "spectra")

In [ ]:
F_start, F_stop = None, None  # frequency range shown [Hz]; None = edge of the trace
f_axis = "lin"  # "lin" or "log"

# Rise of each state over all_off, point by point (same sweep points): what that switching step added. 0 dB = no
# pollution; a few dB of scatter is the noise of the averaged traces.
base = traces[traces["state"] == "all_off"].set_index("freq")["level_dbm"]
fig = go.Figure()
fig.add_hline(y=0, line=dict(color=INK, width=1, dash="dash"))
for r in states[states["state"] != "all_off"].itertuples():
    d = traces[traces["state"] == r.state].set_index("freq")["level_dbm"]
    d = in_range((d - base).rename("rise").reset_index(), F_start, F_stop)
    fig.add_scatter(x=d["freq"], y=d["rise"], mode="lines", name=NAMES[r.state],
                    line=dict(color=COLORS.get(r.state, INK), width=1.2),
                    hovertemplate=f"{r.state}<br>%{{x:.4s}}Hz: %{{y:+.1f}} dB<extra></extra>")
fig.update_layout(title="Rise of the spectrum over all_off per state", template="plotly_white", height=500,
                  legend=dict(orientation="h", y=-0.2), hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="frequency", gridcolor=GRID, **freq_axis(f_axis))
fig.update_yaxes(title="rise over all_off [dB]", gridcolor=GRID)
save(fig, "rise")